In [44]:
import os
import numpy as np
import pandas

In [45]:
cilia_all_videos_path = r"D:\Cilia Data\all videos"
cilia_masks_path = r"D:\Cilia Data\converted_masks\full"
video_files = [f for f in os.listdir(cilia_all_videos_path) if f.endswith(".avi")]
mask_files = [f for f in os.listdir(cilia_masks_path) if f.endswith(".npy")]

In [46]:
import cv2
from tqdm import tqdm

first_frames_path = r"D:\Cilia Data\first_frames"
os.makedirs(first_frames_path, exist_ok=True)

for video in tqdm(video_files, desc="Processing videos"):
    video_path = os.path.join(cilia_all_videos_path, video)
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    if ret:
        frame_path = os.path.join(first_frames_path, os.path.splitext(video)[0] + ".png")
        cv2.imwrite(frame_path, frame)
    cap.release()

Processing videos: 100%|██████████| 681/681 [00:06<00:00, 98.77it/s] 


In [47]:
cilia_dict = {}

for video in video_files:
    name = os.path.splitext(video)[0]
    video_path = os.path.join(cilia_all_videos_path, video)
    mask_path = os.path.join(cilia_masks_path, name + ".npy") if name + ".npy" in mask_files else None
    cilia_dict[name] = {"video_path": video_path, "mask_path": mask_path}

cilia_df = pandas.DataFrame.from_dict(cilia_dict, orient="index")
cilia_df.head(5)

,video_path,mask_path
1001-1,D:\Cilia Data\all videos\1001-1.avi,D:\Cilia Data\converted_masks\full\1001-1.npy
1001-13,D:\Cilia Data\all videos\1001-13.avi,D:\Cilia Data\converted_masks\full\1001-13.npy
1001-2,D:\Cilia Data\all videos\1001-2.avi,D:\Cilia Data\converted_masks\full\1001-2.npy
1001-3,D:\Cilia Data\all videos\1001-3.avi,D:\Cilia Data\converted_masks\full\1001-3.npy
1001-5,D:\Cilia Data\all videos\1001-5.avi,D:\Cilia Data\converted_masks\full\1001-5.npy


In [48]:
cilia_df['first_frame_path'] = cilia_df.index.map(lambda name: os.path.join(first_frames_path, name + ".png"))
cilia_df.head(5)

,video_path,mask_path,first_frame_path
1001-1,D:\Cilia Data\all videos\1001-1.avi,D:\Cilia Data\converted_masks\full\1001-1.npy,D:\Cilia Data\first_frames\1001-1.png
1001-13,D:\Cilia Data\all videos\1001-13.avi,D:\Cilia Data\converted_masks\full\1001-13.npy,D:\Cilia Data\first_frames\1001-13.png
1001-2,D:\Cilia Data\all videos\1001-2.avi,D:\Cilia Data\converted_masks\full\1001-2.npy,D:\Cilia Data\first_frames\1001-2.png
1001-3,D:\Cilia Data\all videos\1001-3.avi,D:\Cilia Data\converted_masks\full\1001-3.npy,D:\Cilia Data\first_frames\1001-3.png
1001-5,D:\Cilia Data\all videos\1001-5.avi,D:\Cilia Data\converted_masks\full\1001-5.npy,D:\Cilia Data\first_frames\1001-5.png


In [49]:
def get_frame_size(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if ret:
        return (f"{frame.shape[1]}x{frame.shape[0]}")
    else:
        return None

cilia_df['size'] = cilia_df['video_path'].apply(get_frame_size)
cilia_df.head(5)

,video_path,mask_path,first_frame_path,size
1001-1,D:\Cilia Data\all videos\1001-1.avi,D:\Cilia Data\converted_masks\full\1001-1.npy,D:\Cilia Data\first_frames\1001-1.png,640x480
1001-13,D:\Cilia Data\all videos\1001-13.avi,D:\Cilia Data\converted_masks\full\1001-13.npy,D:\Cilia Data\first_frames\1001-13.png,512x384
1001-2,D:\Cilia Data\all videos\1001-2.avi,D:\Cilia Data\converted_masks\full\1001-2.npy,D:\Cilia Data\first_frames\1001-2.png,640x480
1001-3,D:\Cilia Data\all videos\1001-3.avi,D:\Cilia Data\converted_masks\full\1001-3.npy,D:\Cilia Data\first_frames\1001-3.png,640x480
1001-5,D:\Cilia Data\all videos\1001-5.avi,D:\Cilia Data\converted_masks\full\1001-5.npy,D:\Cilia Data\first_frames\1001-5.png,512x384


In [50]:
def determine_cell_type(name):
    if name.startswith('1'):
        return 'normal'
    elif name.startswith('7'):
        return 'abnormal'
    else:
        return 'indeterminate'

cilia_df['cell_type'] = cilia_df.index.map(determine_cell_type)
cilia_df

,video_path,mask_path,first_frame_path,size,cell_type
1001-1,D:\Cilia Data\all videos\1001-1.avi,D:\Cilia Data\converted_masks\full\1001-1.npy,D:\Cilia Data\first_frames\1001-1.png,640x480,normal
1001-13,D:\Cilia Data\all videos\1001-13.avi,D:\Cilia Data\converted_masks\full\1001-13.npy,D:\Cilia Data\first_frames\1001-13.png,512x384,normal
1001-2,D:\Cilia Data\all videos\1001-2.avi,D:\Cilia Data\converted_masks\full\1001-2.npy,D:\Cilia Data\first_frames\1001-2.png,640x480,normal
1001-3,D:\Cilia Data\all videos\1001-3.avi,D:\Cilia Data\converted_masks\full\1001-3.npy,D:\Cilia Data\first_frames\1001-3.png,640x480,normal
1001-5,D:\Cilia Data\all videos\1001-5.avi,D:\Cilia Data\converted_masks\full\1001-5.npy,D:\Cilia Data\first_frames\1001-5.png,512x384,normal
...,...,...,...,...,...
Cyclo-1-0,D:\Cilia Data\all videos\Cyclo-1-0.avi,None,D:\Cilia Data\first_frames\Cyclo-1-0.png,640x480,indeterminate
normal,D:\Cilia Data\all videos\normal.avi,None,D:\Cilia Data\first_frames\normal.png,128x256,indeterminate
Sag-1-0,D:\Cilia Data\all videos\Sag-1-0.avi,None,D:\Cilia Data\first_frames\Sag-1-0.png,640x480,indeterminate
Sag-1-1,D:\Cilia Data\all videos\Sag-1-1.avi,None,D:\Cilia Data\first_frames\Sag-1-1.png,640x480,indeterminate


In [51]:
def check_mask(mask_path):
    if mask_path and os.path.exists(mask_path):
        mask = np.load(mask_path)
        exposed = np.any(mask == 128)
        overlaying = np.any(mask == 192)
        return exposed, overlaying
    else:
        return None, None

cilia_df['exposed'], cilia_df['overlaying'] = zip(*cilia_df['mask_path'].apply(check_mask))
cilia_df.head(5)

,video_path,mask_path,first_frame_path,size,cell_type,exposed,overlaying
1001-1,D:\Cilia Data\all videos\1001-1.avi,D:\Cilia Data\converted_masks\full\1001-1.npy,D:\Cilia Data\first_frames\1001-1.png,640x480,normal,False,True
1001-13,D:\Cilia Data\all videos\1001-13.avi,D:\Cilia Data\converted_masks\full\1001-13.npy,D:\Cilia Data\first_frames\1001-13.png,512x384,normal,True,False
1001-2,D:\Cilia Data\all videos\1001-2.avi,D:\Cilia Data\converted_masks\full\1001-2.npy,D:\Cilia Data\first_frames\1001-2.png,640x480,normal,True,False
1001-3,D:\Cilia Data\all videos\1001-3.avi,D:\Cilia Data\converted_masks\full\1001-3.npy,D:\Cilia Data\first_frames\1001-3.png,640x480,normal,True,True
1001-5,D:\Cilia Data\all videos\1001-5.avi,D:\Cilia Data\converted_masks\full\1001-5.npy,D:\Cilia Data\first_frames\1001-5.png,512x384,normal,True,True


In [52]:
def generate_description(cell_type):
    if cell_type == 'normal':
        return "an image of a nasal epithelial biopsy showing normal cilia."
    elif cell_type == 'abnormal':
        return "an image of a nasal epithelial biopsy showing abnormal cilia."
    else:
        return "an image of a nasal epithelial biopsy showing cilia."

cilia_df['description'] = cilia_df['cell_type'].apply(generate_description)
cilia_df.head(5)

,video_path,mask_path,first_frame_path,size,cell_type,exposed,overlaying,description
1001-1,D:\Cilia Data\all videos\1001-1.avi,D:\Cilia Data\converted_masks\full\1001-1.npy,D:\Cilia Data\first_frames\1001-1.png,640x480,normal,False,True,an image of a nasal epithelial biopsy showing ...
1001-13,D:\Cilia Data\all videos\1001-13.avi,D:\Cilia Data\converted_masks\full\1001-13.npy,D:\Cilia Data\first_frames\1001-13.png,512x384,normal,True,False,an image of a nasal epithelial biopsy showing ...
1001-2,D:\Cilia Data\all videos\1001-2.avi,D:\Cilia Data\converted_masks\full\1001-2.npy,D:\Cilia Data\first_frames\1001-2.png,640x480,normal,True,False,an image of a nasal epithelial biopsy showing ...
1001-3,D:\Cilia Data\all videos\1001-3.avi,D:\Cilia Data\converted_masks\full\1001-3.npy,D:\Cilia Data\first_frames\1001-3.png,640x480,normal,True,True,an image of a nasal epithelial biopsy showing ...
1001-5,D:\Cilia Data\all videos\1001-5.avi,D:\Cilia Data\converted_masks\full\1001-5.npy,D:\Cilia Data\first_frames\1001-5.png,512x384,normal,True,True,an image of a nasal epithelial biopsy showing ...


In [53]:
cilia_df.to_csv("normal_abnormal.csv", index=False)